# Notebook 03 — PatchCore

## 1. Motivation and hypotheses
Reconstruction models can copy defects and blur normal structure. PatchCore
instead compares local pretrained features against a memory of normal patches.
Hypothesis: local nearest-neighbor distances separate fitting-derived synthetic
edits from normal appearance without a learned reconstruction objective.
This first execution is a live small normal-only CUDA pilot; development and
all-category confirmation remain pending. Earlier notebooks/results are preserved.

## 2. Architecture and mathematics
For frozen CNN activations $F_2(x),F_3(x)$, apply local average pooling and align
the coarser map to the finer grid. Concatenation gives $z_{h,w}\in\mathbb R^{384}$.
The fitting-normal memory is $M=\{z_i\}$. A coreset approximately minimizes
$\max_{z\in M}\min_{m\in M_C}\|Pz-Pm\|_2$, using random projection $P$ and greedy
farthest-first traversal of a capped candidate pool. Search uses original
384-dimensional vectors: $a_{h,w}=\min_{m\in M_C}\|z_{h,w}-m\|_2$.
Continuous patch distances are interpolated into a pixel map. Image scores use
maximum patch distance or a fixed top-1% mean; they are not pixel probabilities.

This is a **simplified PatchCore variant**, not paper equivalence: ResNet18,
direct concatenation, capped candidate-pool projected coreset, no neighborhood
image-score reweighting and initially no Gaussian smoothing. Sources:
[Roth et al., CVPR 2022](https://arxiv.org/abs/2106.08265) and
[torchvision weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18).


In [1]:
from pathlib import Path
import sys,json,time,copy,traceback,subprocess,math
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
import numpy as np,pandas as pd,matplotlib.pyplot as plt,torch,psutil
from anomaly_detection.data import CATEGORIES,sha256
from anomaly_detection.basic_cnn import seed_everything,synthetic_batch
from anomaly_detection.cnn_research import read_partition,require_fitting,load_resized
from anomaly_detection.tracking import record,config_hash
from anomaly_detection.patchcore_research import Resources
torch.set_num_threads(4)
assert torch.cuda.is_available()
seed_everything(2026)
torch.backends.cuda.matmul.allow_tf32=False
print(torch.__version__,torch.cuda.get_device_name(0))
print('GPU GiB:',torch.cuda.get_device_properties(0).total_memory/2**30)
print('RAM GiB:',psutil.virtual_memory().total/2**30)


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\shaik\\.matplotlib\\fontlist-v390.json.matplotlib-lock'


2.11.0+cu128 NVIDIA GeForce RTX 4060 Laptop GPU
GPU GiB: 7.99560546875
RAM GiB: 31.70067596435547


## 3. Dataset and frozen partition verification
The existing loader checks manifest hashes and normal-only provenance. No
official test image or mask is accessed by this pilot. The first 24 fitting
normals form the pilot bank; first eight development normals measure transfer.
## 4. Environment and GPU verification
Use the existing locked CUDA environment. Both ResNet18 and WideResNet50-2
weights are cached; start with the smaller ResNet18 and download neither.
Actual shapes, resources and correctness are measured below. Weight hashes and
the source/environment identities are recorded before fitting.


In [2]:
fit_rows=read_partition('fit','bottle').iloc[:24].copy()
require_fitting(fit_rows)
dev_rows=read_partition('development','bottle').iloc[:8].copy()
print('Pilot fitting normals:',len(fit_rows),'development normals:',len(dev_rows))
metadata=json.loads((ROOT/'reports/dataset_validation.json').read_text())
print(json.dumps(metadata['partitions'],indent=2))


Pilot fitting normals: 24 development normals: 8
{
  "fit": {
    "path": "reports/partitions/seed2026-fit.csv",
    "sha256": "f0549001ec7bbf124b67cb3979d2af64f593a2b5c9f420838f539938b4ade912",
    "images": 2551
  },
  "development": {
    "path": "reports/partitions/seed2026-development.csv",
    "sha256": "e1e49dee7d8111b50409e49ab51a046d6888b3a4f82256a927c6b16ace33a4df",
    "images": 539
  },
  "calibration": {
    "path": "reports/partitions/seed2026-calibration.csv",
    "sha256": "dc46dd58419fc0dd32e20f174efe4e208df49bc25c65bdb06afe2dbd1a418ff6",
    "images": 539
  },
  "test": {
    "path": "reports/partitions/seed2026-test.csv",
    "sha256": "bf873b574776e11d2931d3c9004a2570a5362303f10206f0d2afa271274dcfa2",
    "images": 1725
  }
}


## 5. Preprocessing
Default ResNet18 classification weights resize the short edge to 256 and center
crop 224. For full-image localization, retain the previous models' PIL bilinear
256-square geometry, then call the official weight transform with resize/crop
256. This documented geometry override keeps all borders and official ImageNet
mean/std; it does not use test statistics. Aspect ratios can be distorted.
## 6. Pretrained extraction
All parameters and BatchNorm statistics are frozen, including after `.train()`.
## 7. Layer selection
At 256 square: layer1 is 64x64x64, layer2 is 128x32x32 and layer3 is 256x16x16.
Layer2+layer3 yields 1,024 patches/image; using layer1 would quadruple patch
count and search cost. The initial pair balances detail and semantic context.
## 8. Multi-scale aggregation
Pool 3x3 locally, align by bilinear interpolation and concatenate channels.
## 9. Patch embeddings
Flatten spatial locations, not channels: B,C,H,W → (BHW),C.
## 10. Normal memory
Only category-specific fitting normals supply bank vectors.
## 11. Coreset
Random candidate cap 20,000; Gaussian projection to 32 dimensions; greedy
farthest-first selection. Keep original full-dimensional vectors for scoring.
## 12. Nearest-neighbor search
GPU blocked exhaustive L2, at most 512x2048 temporary distances (4 MiB).
Direct subtraction refines selected distances. Index choices near numerical ties
remain float32-precision limited. This is an efficient PyTorch flat-search
equivalent, avoiding an additional FAISS installation on this Windows environment.
## 13. Pixel maps
Interpolate continuous distances; masks are never bilinearly transformed.
## 14. Image scores
Compare max-patch and top-1%-patch mean on permitted development evidence.
The following is the actual implementation, executed as definitions here.


In [3]:
"""Bounded ResNet18 PatchCore variant, not a paper-equivalent reproduction."""
import math
import hashlib
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torchvision.models import resnet18, ResNet18_Weights


class PatchExtractor(nn.Module):
    """Frozen layer2/layer3 features; whole-image geometry preserves edge defects.

    Official V1 mean/std and torchvision weight transforms are used, with a
    documented crop-size override: inputs are already whole-image 256 squares,
    then resize/crop 256 (identity geometry). Default classification 224 center
    cropping is deliberately avoided. No learned test/normal statistics.
    """
    def __init__(self, pretrained=True):
        super().__init__()
        weights=ResNet18_Weights.IMAGENET1K_V1
        if pretrained:
            path=Path(torch.hub.get_dir())/'checkpoints'/Path(weights.url).name
            if not path.is_file():
                raise FileNotFoundError('Register a weight acquisition before downloading: '+str(path))
            self.weight_sha256=hashlib.sha256(path.read_bytes()).hexdigest()
            if not self.weight_sha256.startswith(path.stem.split('-')[-1]):
                raise ValueError('Cached ImageNet weight hash does not match torchvision identity')
        else:self.weight_sha256='untrained-test-only'
        network=resnet18(weights=weights if pretrained else None)
        self.stem=nn.Sequential(network.conv1,network.bn1,network.relu,network.maxpool)
        self.layer1=network.layer1
        self.layer2=network.layer2
        self.layer3=network.layer3
        self.transform=weights.transforms(crop_size=256,resize_size=256,antialias=True)
        self.requires_grad_(False)
        self.eval()

    def train(self,mode=True):
        # Even parent .train() must not update ImageNet BatchNorm statistics.
        return super().train(False)

    @torch.inference_mode()
    def feature_maps(self,images):
        if images.ndim!=4 or images.shape[1:]!=(3,256,256) or not images.is_floating_point():
            raise ValueError('Expected B,3,256,256 float RGB in [0,1]')
        x=self.transform(images)
        first=self.layer1(self.stem(x))
        second=self.layer2(first)
        third=self.layer3(second)
        return first,second,third

    @torch.inference_mode()
    def forward(self,images):
        _,second,third=self.feature_maps(images)
        return align_features(second,third)


def align_features(shallow,deep):
    """3x3 local average pooling, coarse bilinear alignment, channel concatenation."""
    if shallow.ndim!=4 or deep.ndim!=4 or shallow.shape[0]!=deep.shape[0]:
        raise ValueError('Matching B,C,H,W feature batches required')
    local=F.avg_pool2d(shallow,3,1,1)
    coarse=F.avg_pool2d(deep,3,1,1)
    coarse=F.interpolate(coarse,size=local.shape[-2:],mode='bilinear',align_corners=False)
    return torch.cat([local,coarse],dim=1)


def patch_embeddings(features):
    if features.ndim!=4:raise ValueError('Expected B,C,H,W features')
    return features.permute(0,2,3,1).reshape(-1,features.shape[1]).contiguous()


@torch.inference_mode()
def projected_coreset(embeddings,budget,seed=2026,candidate_limit=20000,projection_dim=32):
    """Random capped candidate pool -> projected farthest-first traversal.

    Returns unique indices into original embeddings plus measured projected
    coverage distances. Search uses original full-dimensional vectors, not the
    projection. The cap is a disclosed approximation to a full-data coreset.
    """
    if embeddings.ndim!=2 or not torch.isfinite(embeddings).all():raise ValueError('Finite N,D matrix required')
    n,d=embeddings.shape
    if not 1<=budget<=min(n,candidate_limit) or projection_dim<1:raise ValueError('Invalid coreset budget')
    generator=torch.Generator().manual_seed(seed)
    candidates=torch.randperm(n,generator=generator)[:candidate_limit].to(embeddings.device)
    projection=torch.randn(d,projection_dim,generator=generator).to(embeddings.device)/math.sqrt(projection_dim)
    reduced=embeddings[candidates].float()@projection
    distances=torch.full((len(candidates),),float('inf'),device=embeddings.device)
    available=torch.ones(len(candidates),dtype=torch.bool,device=embeddings.device)
    chosen=[]
    current=0  # randomized candidate order supplies a seeded initial center
    for _ in range(budget):
        chosen.append(current)
        available[current]=False
        delta=(reduced-reduced[current]).square().sum(1)
        distances=torch.minimum(distances,delta)
        current=int(distances.masked_fill(~available,-1).argmax())
    indices=candidates[torch.tensor(chosen,device=embeddings.device)]
    coverage=dict(candidate_count=len(candidates),projected_radius=float(distances.max().sqrt()),
                  projected_mean_distance=float(distances.sqrt().mean()),projection_dim=projection_dim)
    return indices,coverage


@torch.inference_mode()
def nearest_neighbors(queries,bank,query_chunk=512,bank_chunk=2048):
    """Exhaustive float32 L2 search with a bounded Qchunk x Mchunk matrix.

    GEMM finds the nearest index; direct subtraction refines the selected
    distance, avoiding cancellation in self/near-self distance values. As with
    float32 flat search, almost-tied index choices are numerical-precision limited.
    No Q x M x D tensor or complete image-batch distance matrix is constructed.
    """
    if queries.ndim!=2 or bank.ndim!=2 or queries.shape[1]!=bank.shape[1] or not len(bank):
        raise ValueError('Compatible query and nonempty bank matrices required')
    if query_chunk<1 or bank_chunk<1 or queries.device!=bank.device:raise ValueError('Invalid chunks/device')
    distances=[];indices=[]
    for q in queries.float().split(query_chunk):
        best=torch.full((len(q),),float('inf'),device=q.device)
        best_index=torch.zeros(len(q),dtype=torch.long,device=q.device)
        for start in range(0,len(bank),bank_chunk):
            b=bank[start:start+bank_chunk].float()
            squared=(q.square().sum(1,keepdim=True)+b.square().sum(1)[None]-2*(q@b.T)).clamp_min_(0)
            values,positions=squared.min(1)
            improved=values<best
            best=torch.minimum(best,values)
            best_index=torch.where(improved,positions+start,best_index)
        distances.append((q-bank[best_index]).square().sum(1).sqrt())
        indices.append(best_index)
    return torch.cat(distances),torch.cat(indices)


def pixel_maps(patch_distances,shape=(256,256)):
    if patch_distances.ndim!=3:raise ValueError('Expected B,Hpatch,Wpatch')
    return F.interpolate(patch_distances[:,None],size=shape,mode='bilinear',align_corners=False)[:,0]


def image_scores(patch_distances,aggregation='max'):
    flat=patch_distances.flatten(1)
    if aggregation=='max':return flat.amax(1)
    if aggregation=='top1pct':return flat.topk(max(1,math.ceil(flat.shape[1]*.01)),dim=1).values.mean(1)
    raise ValueError('Expected max or top1pct')


def save_bank(path,bank,indices,category,config,provenance):
    if path.exists():raise FileExistsError(path)
    torch.save(dict(bank=bank.cpu(),indices=indices.cpu(),category=category,config=config,provenance=provenance),path)


def load_bank(path,category,device='cpu'):
    saved=torch.load(path,map_location='cpu',weights_only=True)
    if saved['category']!=category:raise ValueError('Category-specific bank mismatch')
    return saved['bank'].to(device),saved


## 15. Initial verification and measured pilot
Hypothesis: 24 fitting normals and a 512-patch coreset fit safely, exact-search
fixtures agree with float64 reference distances, self-search returns zero and
the score map has the declared shape. These checks precede all-category work.
The pilot is a feasibility experiment, not an official detection benchmark.


In [4]:
identifier='03-pilot-resnet18-bottle-s2026-v1'
config=dict(backbone='resnet18',weights='IMAGENET1K_V1',layers=['layer2','layer3'],
    resolution=256,bank_budget=512,candidate_limit=20000,projection_dim=32,
    query_chunk=512,bank_chunk=2048,seed=2026,normalize=False)
local=ROOT/'outputs/patchcore'/identifier
local.mkdir(parents=True,exist_ok=True)
report=ROOT/'reports/patchcore';report.mkdir(parents=True,exist_ok=True)
entry=dict(experiment_id=identifier+'-start',model_family='PatchCoreVariant',categories=['bottle'],
    hypothesis='A cached frozen ResNet18 with a 512-patch projected coreset supports bounded exact search and finite anomaly maps.',
    preprocessing='Whole PIL RGB bilinear 256 + official V1 transform with identity resize/crop 256 geometry',
    hyperparameters=config,seed=2026,
    environment=dict(torch=torch.__version__,gpu=torch.cuda.get_device_name(0),ram_bytes=psutil.virtual_memory().total,
        git_sha=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip(),
        source_sha256={p:sha256(ROOT/p) for p in ['src/anomaly_detection/patchcore.py','src/anomaly_detection/patchcore_research.py']},
        lock_sha256=sha256(ROOT/'requirements.lock.txt'),partitions=metadata['partitions']),
    training={},metrics={},calibration={},paths=dict(local=local.relative_to(ROOT).as_posix()),
    interpretation='Reserved before normal-only pilot; no official test access',decision='Measure feasibility before expansion',status='started')
record(ROOT/'experiments',entry)
try:
    extractor=PatchExtractor().cuda()
    print('Weight SHA256:',extractor.weight_sha256)
    print('Transform:',extractor.transform)
    with Resources() as resource:
        fitting=load_resized(fit_rows,256).cuda().float()/255
        development=load_resized(dev_rows,256).cuda().float()/255
        tick=time.perf_counter()
        features=extractor(fitting)
        torch.cuda.synchronize();extraction_seconds=time.perf_counter()-tick
        shapes=[list(f.shape) for f in extractor.feature_maps(fitting[:1])]
        embeddings=patch_embeddings(features)
        tick=time.perf_counter()
        indices,coverage=projected_coreset(embeddings,config['bank_budget'],config['seed'])
        bank=embeddings[indices].clone()
        torch.cuda.synchronize();coreset_seconds=time.perf_counter()-tick
        self_distances,_=nearest_neighbors(bank[:16],bank)
        assert self_distances.eq(0).all()
        g=torch.Generator().manual_seed(3)
        fixture_bank=torch.randn(37,16,generator=g).cuda()
        fixture_query=torch.randn(13,16,generator=g).cuda()
        actual,_=nearest_neighbors(fixture_query,fixture_bank,7,11)
        reference=torch.cdist(fixture_query.double(),fixture_bank.double(),compute_mode='donot_use_mm_for_euclid_dist').min(1).values
        torch.testing.assert_close(actual.double(),reference,rtol=1e-6,atol=1e-6)
        dev_features=extractor(development)
        tick=time.perf_counter()
        distances,_=nearest_neighbors(patch_embeddings(dev_features),bank)
        torch.cuda.synchronize();search_seconds=time.perf_counter()-tick
        patch_maps=distances.reshape(len(development),32,32)
        maps=pixel_maps(patch_maps)
        assert maps.shape==(8,256,256) and torch.isfinite(maps).all() and maps.ge(0).all()
        for _ in range(20):nearest_neighbors(patch_embeddings(dev_features[:1]),bank)
        times=[]
        for _ in range(50):
            tick=time.perf_counter();nearest_neighbors(patch_embeddings(dev_features[:1]),bank)
            torch.cuda.synchronize();times.append((time.perf_counter()-tick)*1000)
        altered=development[:1].clone()
        altered[:,:,96:128,96:128]=1-altered[:,:,96:128,96:128]
        altered_distances,_=nearest_neighbors(patch_embeddings(extractor(altered)),bank)
        altered_map=pixel_maps(altered_distances.reshape(1,32,32))
    fitting_counts={c:len(read_partition('fit',c)) for c in CATEGORIES}
    result=dict(configuration=config,weight_sha256=extractor.weight_sha256,
        weight_url=ResNet18_Weights.IMAGENET1K_V1.url,
        extractor_parameters=sum(p.numel() for p in extractor.parameters()),
        trainable_parameters=sum(p.numel() for p in extractor.parameters() if p.requires_grad),
        feature_shapes=shapes,embedding_shape=list(embeddings.shape),patches_per_image=1024,
        extraction_seconds=extraction_seconds,coreset_seconds=coreset_seconds,
        search_8_images_seconds=search_seconds,search_p50_ms=float(np.median(times)),
        search_p95_ms=float(np.quantile(times,.95)),coverage=coverage,
        resources=resource.result,bank_bytes=bank.numel()*bank.element_size(),
        feature_bytes_per_image=1024*384*4,
        all_category_fitting_images=fitting_counts,
        expected_feature_cache_bytes={c:n*1024*384*4 for c,n in fitting_counts.items()},
        projected_three_seed_final_extraction_seconds=extraction_seconds/24*sum(fitting_counts.values()),
        coreset_cost_caveat='Pilot bank 512; candidate cap and selected bank budget affect final cost. Deterministic extraction runs once, not three times.',
        correctness=dict(float64_fixture=True,zero_self_distances=True,finite_nonnegative_maps=True,map_shape=list(maps.shape)))
    torch.save(dict(bank=bank.cpu(),indices=indices.cpu(),config=config),local/'pilot-bank.pt')
    torch.save(dict(normal=development.cpu(),maps=maps.cpu(),altered=altered.cpu(),altered_map=altered_map.cpu()),local/'pilot-panels.pt')
    with (report/(identifier+'-result.json')).open('x') as stream:json.dump(result,stream,indent=2)
    event=copy.deepcopy(entry)
    event.update(experiment_id=identifier+'-result',status='completed',metrics=result,
        decision='Pilot correctness passed; assess measured workload before bounded development',
        interpretation='Feasibility evidence only; synthetic map example is not a real-defect benchmark.')
    record(ROOT/'experiments',event)
    print(json.dumps(result,indent=2))
except BaseException as exc:
    (local/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8')
    event=copy.deepcopy(entry)
    event.update(experiment_id=identifier+'-failure',status='failed',decision='preserve and investigate',interpretation=str(exc))
    record(ROOT/'experiments',event)
    raise


Weight SHA256: f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec
Transform: ImageClassification(
    crop_size=[256]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


{
  "configuration": {
    "backbone": "resnet18",
    "weights": "IMAGENET1K_V1",
    "layers": [
      "layer2",
      "layer3"
    ],
    "resolution": 256,
    "bank_budget": 512,
    "candidate_limit": 20000,
    "projection_dim": 32,
    "query_chunk": 512,
    "bank_chunk": 2048,
    "seed": 2026,
    "normalize": false
  },
  "weight_sha256": "f37072fd47e89c5e827621c5baffa7500819f7896bbacec160b1a16c560e07ec",
  "weight_url": "https://download.pytorch.org/models/resnet18-f37072fd.pth",
  "extractor_parameters": 2782784,
  "trainable_parameters": 0,
  "feature_shapes": [
    [
      1,
      64,
      64,
      64
    ],
    [
      1,
      128,
      32,
      32
    ],
    [
      1,
      256,
      16,
      16
    ]
  ],
  "embedding_shape": [
    24576,
    384
  ],
  "patches_per_image": 1024,
  "extraction_seconds": 0.442961100001412,
  "coreset_seconds": 0.8132004999970377,
  "search_8_images_seconds": 0.030620199999248143,
  "search_p50_ms": 4.397949998747208,
  "searc

### Pilot interpretation and decision
Use the measured output above to decide feasibility. Self-distance and float64
fixture checks test implementation correctness, not detection quality. Coreset
coverage is measured only in projected candidate space; it does not guarantee
coverage of every original patch. The synthetic edit below is a map sanity check.
Licensed image panels remain local; public outputs retain measurements.


In [5]:
fig,axes=plt.subplots(2,2,figsize=(8,8))
limit=float(torch.cat([maps[:1],altered_map]).max())
for i,(images,heatmaps) in enumerate([(development[:1],maps[:1]),(altered,altered_map)]):
    axes[i,0].imshow(images[0].cpu().permute(1,2,0));axes[i,0].axis('off')
    axes[i,1].imshow(heatmaps[0].cpu(),vmin=0,vmax=limit);axes[i,1].axis('off')
axes[0,0].set_title('Real development normal')
axes[1,0].set_title('Synthetic edit, not official defect')
plt.tight_layout();plt.show()


## 16. Development experiments
Predeclared bottle/cable/carpet/tile; bank budgets 256/1024/2048; max versus
top-1%-patch mean. One frozen backbone/layer pair, one blocked search backend.
Normal development diagnostics and explicitly synthetic inner-fitting holdout
probes support selection. All variants, timings and decisions must be retained.
## 17. Candidate selection
Four-category mean (proxy image AUROC + proxy pixel AP)/2, then latency/bank size
for exact ties. No official test-label tuning. Selection is pending.
## 18. Threshold calibration
Reserved normal partition only; image higher q95 and separate pixel q99.5.
Report counts, strict > decisions, test FPR and uncertainty. Pending freeze.
## 19. Frozen configuration
Freeze complete source, weight, coreset, preprocessing, scoring and calibration
identities before confirmation. No freeze or final evaluation is claimed yet.
## 20. All-category evaluation
Pending the measured gate, then all 15 categories, preferably three coreset seeds.
Cache compatible deterministic features instead of extracting them repeatedly.
## 21. Anomaly maps and overlays
Final original/heatmap/overlay/native-ground-truth panels are pending evaluation.
## 22. Failure analysis
Investigate small scratches, texture, structure, memory coverage and threshold
instability using observed data. Do not infer causes from unseen results.
## 23. Runtime and GPU/CPU memory
Pilot measures extraction, coreset, search, RSS sampling and CUDA peaks. Final
latency will use 50 warmups/200 batch-one timed runs, plus separate end-to-end cost.
## 24. Comparison with CNN and Autoencoder
Pending final results. Read exact stored summaries and category-matched scores;
do not substitute rounded references or select a winner on AUROC alone.
## 25. Conclusions and limitations
Only implementation/pilot evidence exists at this chronological point. The
variant omits paper reweighting and smoothing, caps coreset candidates and uses
ImageNet transfer with a localization geometry override. The public notebook
removes only licensed image-panel outputs; the full executed pilot stays local.
No SuperSimpleNet work is part of this milestone.
